
# Forudsigelse af Stressniveau med KNN (Student Lifestyle Dataset)

Denne notebook indeholder hele workflowet til jeres præsentation **mandag**:
1. Indlæsning og hurtig EDA
2. Databehandling (oprydning, encoding, skalering)
3. Visualiseringer (diagrammer og figurer)
4. Udvælgelse af features
5. KNN-model + hyperparametertuning
6. Evaluering og eksempel-forudsigelser

> **Mål:** Forudsige `Stress_Level` ud fra de øvrige faktorer.


## 0. Opsætning og import

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option("display.max_columns", None)


## 1. Indlæs data

In [ ]:

data_path = "student_lifestyle_dataset.csv"
df = pd.read_csv(data_path)
df.head()


## 2. Hurtig EDA (overblik over data)

In [ ]:

display(df.info())
df.describe()


## 3. Databehandling (oprydning og encoding)

In [ ]:

if "Student_ID" in df.columns:
    df = df.drop(columns=["Student_ID"])

encoder = LabelEncoder()
df["Stress_Level"] = encoder.fit_transform(df["Stress_Level"])

missing = df.isna().sum()
missing


## 4. Visualiseringer (overvejelser og mønstre)


Nedenfor tegnes enkelte **scatter plots** for at vise relationer mellem centrale faktorer og `Stress_Level`.


In [ ]:

plt.figure()
plt.scatter(df["Study_Hours_Per_Day"], df["Stress_Level"])
plt.xlabel("Studietimer pr. dag")
plt.ylabel("Stress_Level (encoded)")
plt.title("Studietimer pr. dag vs. Stress_Level")
plt.show()


In [ ]:

plt.figure()
plt.scatter(df["Sleep_Hours_Per_Day"], df["Stress_Level"])
plt.xlabel("Søvntimer pr. dag")
plt.ylabel("Stress_Level (encoded)")
plt.title("Søvntimer pr. dag vs. Stress_Level")
plt.show()


In [ ]:

plt.figure()
plt.scatter(df["Physical_Activity_Hours_Per_Day"], df["Stress_Level"])
plt.xlabel("Motion pr. dag")
plt.ylabel("Stress_Level (encoded)")
plt.title("Motion pr. dag vs. Stress_Level")
plt.show()


In [ ]:

plt.figure()
plt.scatter(df["Social_Hours_Per_Day"], df["Stress_Level"])
plt.xlabel("Sociale timer pr. dag")
plt.ylabel("Stress_Level (encoded)")
plt.title("Sociale timer pr. dag vs. Stress_Level")
plt.show()


In [ ]:

plt.figure()
plt.scatter(df["Extracurricular_Hours_Per_Day"], df["Stress_Level"])
plt.xlabel("Ekstracurriculære timer pr. dag")
plt.ylabel("Stress_Level (encoded)")
plt.title("Ekstracurriculære timer pr. dag vs. Stress_Level")
plt.show()


In [ ]:

plt.figure()
plt.scatter(df["GPA"], df["Stress_Level"])
plt.xlabel("GPA")
plt.ylabel("Stress_Level (encoded)")
plt.title("GPA vs. Stress_Level")
plt.show()


## 5. Udvælgelse af features

In [ ]:

selected_features = [
    "Study_Hours_Per_Day",
    "Sleep_Hours_Per_Day",
    "Physical_Activity_Hours_Per_Day",
    "Social_Hours_Per_Day",
    "Extracurricular_Hours_Per_Day",
    "GPA",
]

X = df[selected_features]
y = df["Stress_Level"]

X.head()


## 6. Train/Test split og skalering

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train.shape, X_test.shape


## 7. Baseline KNN-model

In [ ]:

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)

print("Accuracy (baseline k=5):", accuracy_score(y_test, y_pred))
print("\nClassification report (baseline):\n", classification_report(y_test, y_pred, target_names=encoder.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure()
plt.imshow(cm, aspect="auto")
plt.title("Confusion Matrix (baseline k=5)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()


## 8. Hyperparametertuning (GridSearchCV)

In [ ]:

param_grid = {
    "n_neighbors": list(range(1, 26, 2)),
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}

grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)
grid.fit(X_train_scaled, y_train)

print("Bedste parametre:", grid.best_params_)
print("Bedste CV-accuracy:", grid.best_score_)

best_knn = grid.best_estimator_
y_pred_best = best_knn.predict(X_test_scaled)

print("\nTest Accuracy (best model):", accuracy_score(y_test, y_pred_best))
print("\nClassification report (best model):\n", classification_report(y_test, y_pred_best, target_names=encoder.classes_))

cm_best = confusion_matrix(y_test, y_pred_best)
plt.figure()
plt.imshow(cm_best, aspect="auto")
plt.title("Confusion Matrix (best KNN)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()


## 9. Eksempel: Forudsig et stressniveau for en ny elev

In [ ]:

new_student = pd.DataFrame([{
    "Study_Hours_Per_Day": 6.0,
    "Sleep_Hours_Per_Day": 6.5,
    "Physical_Activity_Hours_Per_Day": 3.0,
    "Social_Hours_Per_Day": 2.0,
    "Extracurricular_Hours_Per_Day": 1.5,
    "GPA": 3.0,
}])

new_student_scaled = scaler.transform(new_student)
pred_encoded = best_knn.predict(new_student_scaled)
pred_label = encoder.inverse_transform(pred_encoded)

print("Forudsagt Stress_Level:", pred_label[0])


## 10. (Valgfrit) Gem model og scaler til senere brug

In [ ]:

import joblib

joblib.dump(best_knn, "knn_stress_model.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(encoder, "label_encoder.joblib")

print("Gemte: knn_stress_model.joblib, scaler.joblib, label_encoder.joblib")
